In [1]:
import re
import glob
from pathlib import Path
from datetime import date

import numpy as np
import xarray as xr
import pandas as pd


# ============================================================
# Settings
# ============================================================

DATA_DIR = Path("/nird/datalake/NS11071K/users/yongyub/Observation/GlobColour")

# Newly created valid RECCAP mask based on zero-filled percentage threshold
VALID_MASK_FILE = Path(
    "/nird/datalake/NS11071K/users/yongyub/Observation/GlobColour/"
    "GlobColour_PP_grid_validity_RECCAP2_all_regions/"
    "valid_mask_zero30/"
    "RECCAP2_all_regions_valid_mask_zero30_on_PPgrid.nc"
)

VALID_MASK_VAR = "valid_region_mask"

YEAR_START = 1998
YEAR_END   = 2025

OUT_DIR = DATA_DIR / "RECCAP2_all_regions_valid_zero30_region_mean_PP"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / (
    f"GlobColour_PP_RECCAP2_all_regions_valid_zero30_area_mean_"
    f"{YEAR_START}_{YEAR_END}.nc"
)

LOG_FILE = OUT_DIR / (
    f"GlobColour_PP_RECCAP2_all_regions_valid_zero30_zero_filled_log_"
    f"{YEAR_START}_{YEAR_END}.csv"
)

SUMMARY_FILE = OUT_DIR / (
    f"GlobColour_PP_RECCAP2_all_regions_valid_zero30_region_summary_"
    f"{YEAR_START}_{YEAR_END}.csv"
)

VERBOSE_ZERO_FILL = False
PROGRESS_EVERY = 12


# ============================================================
# Helper functions
# ============================================================

def parse_pp_filename(path):
    """
    Parse filename like PP_1998_01.nc.
    """
    name = Path(path).name
    m = re.match(r"PP_(\d{4})_(\d{2})\.nc$", name)

    if m is None:
        return None

    return int(m.group(1)), int(m.group(2))


def to_str_array(values):
    """
    Convert xarray/netCDF string or byte-string values to normal Python strings.
    """
    out = []

    for v in values:
        if isinstance(v, bytes):
            out.append(v.decode("utf-8"))
        else:
            out.append(str(v))

    return out


def get_fill_values(da, parent_da=None):
    """
    Collect _FillValue and missing_value from a DataArray and parent variable.
    """
    values = []
    attrs_list = [da.attrs]

    if parent_da is not None:
        attrs_list.append(parent_da.attrs)

    for attrs in attrs_list:
        for key in ["_FillValue", "missing_value"]:
            if key not in attrs:
                continue

            arr = np.asarray(attrs[key]).ravel()

            for x in arr:
                try:
                    x = float(x)
                except Exception:
                    continue

                if np.isfinite(x):
                    values.append(x)

    return sorted(set(values))


def build_region_metadata(mask_ds, region_codes):
    """
    Build metadata table for retained regions.

    Regions with no retained valid grid cells are excluded from region_codes.
    """
    region_codes = np.asarray(region_codes, dtype=np.int16)

    if "region" not in mask_ds.coords:
        return pd.DataFrame(
            {
                "region": region_codes,
                "region_name": [f"region {int(c)}" for c in region_codes],
                "basin": [""] * len(region_codes),
                "original_region_id": [-999] * len(region_codes),
                "n_total_cells_original_mask": [-999] * len(region_codes),
                "n_valid_cells_threshold_mask": [-999] * len(region_codes),
                "valid_fraction_percent_threshold_mask": [np.nan] * len(region_codes),
                "region_fully_excluded": [False] * len(region_codes),
            }
        )

    all_regions = mask_ds["region"].values.astype(int)

    meta = pd.DataFrame({"region": all_regions})

    if "region_name" in mask_ds:
        meta["region_name"] = to_str_array(mask_ds["region_name"].values)
    else:
        meta["region_name"] = [f"region {int(c)}" for c in all_regions]

    if "basin" in mask_ds:
        meta["basin"] = to_str_array(mask_ds["basin"].values)
    else:
        meta["basin"] = [""] * len(all_regions)

    if "original_region_id" in mask_ds:
        meta["original_region_id"] = mask_ds["original_region_id"].values.astype(int)
    else:
        meta["original_region_id"] = -999

    if "n_total_cells" in mask_ds:
        meta["n_total_cells_original_mask"] = mask_ds["n_total_cells"].values.astype(int)
    else:
        meta["n_total_cells_original_mask"] = -999

    if "n_valid_cells" in mask_ds:
        meta["n_valid_cells_threshold_mask"] = mask_ds["n_valid_cells"].values.astype(int)
    else:
        meta["n_valid_cells_threshold_mask"] = -999

    if "valid_fraction_percent" in mask_ds:
        meta["valid_fraction_percent_threshold_mask"] = mask_ds["valid_fraction_percent"].values.astype(float)
    else:
        meta["valid_fraction_percent_threshold_mask"] = np.nan

    if "region_fully_excluded" in mask_ds:
        meta["region_fully_excluded"] = mask_ds["region_fully_excluded"].values.astype(bool)
    else:
        meta["region_fully_excluded"] = False

    meta = meta.loc[meta["region"].isin(region_codes)].copy()

    # Preserve region_codes order
    meta["region"] = pd.Categorical(
        meta["region"],
        categories=[int(c) for c in region_codes],
        ordered=True,
    )

    meta = meta.sort_values("region").reset_index(drop=True)
    meta["region"] = meta["region"].astype(int)

    return meta


# ============================================================
# Find PP files
# ============================================================

all_files = sorted(glob.glob(str(DATA_DIR / "PP_*.nc")))

records = []

for f in all_files:
    parsed = parse_pp_filename(f)

    if parsed is None:
        continue

    year, month = parsed

    if YEAR_START <= year <= YEAR_END:
        records.append(
            {
                "file": f,
                "year": year,
                "month": month,
                "time": np.datetime64(f"{year:04d}-{month:02d}-15"),
            }
        )

records = sorted(records, key=lambda x: (x["year"], x["month"]))
n_time = len(records)

print(f"Number of PP files found: {n_time}")

expected_n = (YEAR_END - YEAR_START + 1) * 12
if n_time != expected_n:
    print(f"WARNING: expected {expected_n} files, but found {n_time}")

if n_time == 0:
    raise FileNotFoundError("No PP files found.")

for r in records[:3]:
    print("first files:", r["file"])

for r in records[-3:]:
    print("last files :", r["file"])


# ============================================================
# Read valid mask
# ============================================================

mask_ds = xr.open_dataset(VALID_MASK_FILE)

if VALID_MASK_VAR not in mask_ds:
    raise KeyError(f"{VALID_MASK_VAR} not found in {VALID_MASK_FILE}")

valid_region_mask = mask_ds[VALID_MASK_VAR].values.astype(np.int16)

pp_lat = mask_ds["latitude"].values.astype(np.float64)
pp_lon = mask_ds["longitude"].values.astype(np.float64)

nlat = len(pp_lat)
nlon = len(pp_lon)

print("\nPP grid from valid mask:")
print("lat:", pp_lat.shape, pp_lat.min(), pp_lat.max())
print("lon:", pp_lon.shape, pp_lon.min(), pp_lon.max())

region_flat = valid_region_mask.ravel()
selected_flat = region_flat > 0

region_codes = np.array(
    sorted([int(v) for v in np.unique(region_flat) if v > 0]),
    dtype=np.int16,
)

n_region = len(region_codes)

print("\nRetained region codes:")
print(region_codes)
print(f"Number of retained regions: {n_region}")

if n_region == 0:
    raise ValueError("No retained valid regions found in valid_region_mask.")

region_meta = build_region_metadata(mask_ds, region_codes)

print("\nRetained region metadata:")
print(region_meta.to_string(index=False))


# ============================================================
# Check PP grid consistency
# ============================================================

with xr.open_dataset(records[0]["file"], decode_times=False, mask_and_scale=False) as ds0:
    src_lat = ds0["latitude"].values.astype(np.float64)
    src_lon = ds0["longitude"].values.astype(np.float64)

if len(src_lat) != nlat or len(src_lon) != nlon:
    raise ValueError("PP grid shape does not match valid mask grid.")

if not np.allclose(src_lat, pp_lat):
    raise ValueError("PP latitude coordinate does not match valid mask latitude.")

if not np.allclose(src_lon, pp_lon):
    raise ValueError("PP longitude coordinate does not match valid mask longitude.")


# ============================================================
# Area weights and static denominator
# ============================================================

lat_weight = np.cos(np.deg2rad(pp_lat)).astype(np.float64)
weight_flat = np.repeat(lat_weight, nlon)

max_code = int(region_codes.max())

region_total_weight = np.bincount(
    region_flat[selected_flat],
    weights=weight_flat[selected_flat],
    minlength=max_code + 1,
)

region_total_grid_count = np.bincount(
    region_flat[selected_flat],
    minlength=max_code + 1,
)

# Keep only regions with positive denominator.
region_codes = np.array(
    [int(c) for c in region_codes if region_total_weight[int(c)] > 0],
    dtype=np.int16,
)

n_region = len(region_codes)

region_meta = region_meta.loc[
    region_meta["region"].isin(region_codes.astype(int))
].copy().reset_index(drop=True)

print("\nRegions used for averaging:")
print(region_meta[["region", "basin", "original_region_id", "region_name"]].to_string(index=False))


# ============================================================
# Allocate output arrays
# ============================================================

times = np.array([r["time"] for r in records])

# Main output: invalid monthly PP values are set to zero.
pp_mean = np.full((n_time, n_region), np.nan, dtype=np.float32)

# Sensitivity output: invalid monthly PP values are excluded from denominator.
pp_valid_only_mean = np.full((n_time, n_region), np.nan, dtype=np.float32)

# Diagnostics
negative_grid_count = np.zeros((n_time, n_region), dtype=np.int32)
nan_grid_count = np.zeros((n_time, n_region), dtype=np.int32)
fill_value_grid_count = np.zeros((n_time, n_region), dtype=np.int32)
zero_filled_grid_count = np.zeros((n_time, n_region), dtype=np.int32)

negative_area_fraction = np.zeros((n_time, n_region), dtype=np.float32)
nan_area_fraction = np.zeros((n_time, n_region), dtype=np.float32)
fill_value_area_fraction = np.zeros((n_time, n_region), dtype=np.float32)
zero_filled_area_fraction = np.zeros((n_time, n_region), dtype=np.float32)

monthly_valid_area_fraction = np.full((n_time, n_region), np.nan, dtype=np.float32)

log_rows = []


# ============================================================
# Loop over monthly PP files
# ============================================================

for it, rec in enumerate(records):
    f = rec["file"]
    year = rec["year"]
    month = rec["month"]

    if (it % PROGRESS_EVERY == 0) or (it == n_time - 1):
        print(f"[{it+1:03d}/{n_time:03d}] processing {year}-{month:02d}")

    with xr.open_dataset(f, decode_times=False, mask_and_scale=False) as ds:
        da = ds["PP"].isel(time=0)

        arr = da.values.astype(np.float64)

        if arr.shape != valid_region_mask.shape:
            raise ValueError(f"PP array shape does not match valid mask for file: {f}")

        arr_flat = arr.ravel()

        fill_values = get_fill_values(da, parent_da=ds["PP"])

        selected = selected_flat

        finite = np.isfinite(arr_flat)

        is_nan = selected & (~finite)

        is_fill = np.zeros(arr_flat.shape, dtype=bool)

        for fv in fill_values:
            is_fill |= selected & finite & np.isclose(arr_flat, fv)

        # True negative values, excluding fill/missing values
        is_negative = selected & finite & (arr_flat < 0.0) & (~is_fill)

        zero_filled = is_nan | is_fill | is_negative

        # ----------------------------------------------------
        # Main method: zero-fill invalid monthly values
        # ----------------------------------------------------
        arr_clean = arr_flat.copy()
        arr_clean[zero_filled] = 0.0
        arr_clean[~np.isfinite(arr_clean)] = 0.0

        num_zero = np.bincount(
            region_flat[selected],
            weights=arr_clean[selected] * weight_flat[selected],
            minlength=max_code + 1,
        )

        # ----------------------------------------------------
        # Sensitivity method: valid-only monthly averaging
        # ----------------------------------------------------
        valid_month = selected & (~zero_filled)

        num_valid_only = np.bincount(
            region_flat[valid_month],
            weights=arr_flat[valid_month] * weight_flat[valid_month],
            minlength=max_code + 1,
        )

        denom_valid_only = np.bincount(
            region_flat[valid_month],
            weights=weight_flat[valid_month],
            minlength=max_code + 1,
        )

        # ----------------------------------------------------
        # Diagnostics
        # ----------------------------------------------------
        neg_count = np.bincount(
            region_flat[is_negative],
            minlength=max_code + 1,
        )

        nan_count = np.bincount(
            region_flat[is_nan],
            minlength=max_code + 1,
        )

        fill_count = np.bincount(
            region_flat[is_fill],
            minlength=max_code + 1,
        )

        zero_count = np.bincount(
            region_flat[zero_filled],
            minlength=max_code + 1,
        )

        neg_weight = np.bincount(
            region_flat[is_negative],
            weights=weight_flat[is_negative],
            minlength=max_code + 1,
        )

        nan_weight = np.bincount(
            region_flat[is_nan],
            weights=weight_flat[is_nan],
            minlength=max_code + 1,
        )

        fill_weight = np.bincount(
            region_flat[is_fill],
            weights=weight_flat[is_fill],
            minlength=max_code + 1,
        )

        zero_weight = np.bincount(
            region_flat[zero_filled],
            weights=weight_flat[zero_filled],
            minlength=max_code + 1,
        )

        valid_weight = np.bincount(
            region_flat[valid_month],
            weights=weight_flat[valid_month],
            minlength=max_code + 1,
        )

        # ----------------------------------------------------
        # Fill outputs
        # ----------------------------------------------------
        for ir, code in enumerate(region_codes):
            code = int(code)

            denom = region_total_weight[code]

            if denom <= 0:
                continue

            pp_mean[it, ir] = num_zero[code] / denom

            if denom_valid_only[code] > 0:
                pp_valid_only_mean[it, ir] = num_valid_only[code] / denom_valid_only[code]

            negative_grid_count[it, ir] = int(neg_count[code])
            nan_grid_count[it, ir] = int(nan_count[code])
            fill_value_grid_count[it, ir] = int(fill_count[code])
            zero_filled_grid_count[it, ir] = int(zero_count[code])

            negative_area_fraction[it, ir] = neg_weight[code] / denom
            nan_area_fraction[it, ir] = nan_weight[code] / denom
            fill_value_area_fraction[it, ir] = fill_weight[code] / denom
            zero_filled_area_fraction[it, ir] = zero_weight[code] / denom
            monthly_valid_area_fraction[it, ir] = valid_weight[code] / denom

            if zero_count[code] > 0:
                row = region_meta.loc[region_meta["region"] == code]

                if len(row) > 0:
                    region_name = str(row["region_name"].iloc[0])
                    basin = str(row["basin"].iloc[0])
                    original_region_id = int(row["original_region_id"].iloc[0])
                else:
                    region_name = f"region {code}"
                    basin = ""
                    original_region_id = -999

                log_rows.append(
                    {
                        "time": str(rec["time"]),
                        "year": year,
                        "month": month,
                        "region": code,
                        "basin": basin,
                        "original_region_id": original_region_id,
                        "region_name": region_name,
                        "negative_grid_count": int(neg_count[code]),
                        "nan_grid_count": int(nan_count[code]),
                        "fill_value_grid_count": int(fill_count[code]),
                        "zero_filled_grid_count": int(zero_count[code]),
                        "negative_area_fraction": float(neg_weight[code] / denom),
                        "nan_area_fraction": float(nan_weight[code] / denom),
                        "fill_value_area_fraction": float(fill_weight[code] / denom),
                        "zero_filled_area_fraction": float(zero_weight[code] / denom),
                        "monthly_valid_area_fraction": float(valid_weight[code] / denom),
                    }
                )

                if VERBOSE_ZERO_FILL:
                    print(
                        f"    zero-filled: {year}-{month:02d}, "
                        f"region {code:02d}, {region_name}, "
                        f"negative={int(neg_count[code])}, "
                        f"NaN={int(nan_count[code])}, "
                        f"fill={int(fill_count[code])}, "
                        f"total={int(zero_count[code])}, "
                        f"area_frac={zero_weight[code] / denom:.3f}"
                    )


# ============================================================
# Region metadata arrays
# ============================================================

region_name_list = region_meta["region_name"].astype(str).tolist()
basin_list = region_meta["basin"].astype(str).tolist()
original_region_id_list = region_meta["original_region_id"].astype(int).tolist()

n_total_cells_original_mask = region_meta["n_total_cells_original_mask"].astype(int).values
n_valid_cells_threshold_mask = region_meta["n_valid_cells_threshold_mask"].astype(int).values
valid_fraction_percent_threshold_mask = (
    region_meta["valid_fraction_percent_threshold_mask"].astype(float).values
)

region_name_attr = ", ".join(
    [
        f"{int(code)}.{name}"
        for code, name in zip(region_codes, region_name_list)
    ]
)

region_source_attr = ", ".join(
    [
        f"{int(code)}.{basin}:{int(orig)}.{name}"
        for code, basin, orig, name in zip(
            region_codes,
            basin_list,
            original_region_id_list,
            region_name_list,
        )
    ]
)


# ============================================================
# Save summary CSV
# ============================================================

summary_rows = []

for ir, code in enumerate(region_codes):
    code = int(code)

    summary_rows.append(
        {
            "region": code,
            "basin": basin_list[ir],
            "original_region_id": original_region_id_list[ir],
            "region_name": region_name_list[ir],
            "n_total_cells_original_mask": int(n_total_cells_original_mask[ir]),
            "n_valid_cells_threshold_mask": int(n_valid_cells_threshold_mask[ir]),
            "valid_fraction_percent_threshold_mask": float(valid_fraction_percent_threshold_mask[ir]),
            "n_cells_used_for_regional_mean": int(region_total_grid_count[code]),
            "area_weight_used_for_regional_mean": float(region_total_weight[code]),
            "mean_zero_filled_area_fraction_over_time": float(
                np.nanmean(zero_filled_area_fraction[:, ir])
            ),
            "mean_monthly_valid_area_fraction_over_time": float(
                np.nanmean(monthly_valid_area_fraction[:, ir])
            ),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_FILE, index=False)

print(f"\nSaved summary: {SUMMARY_FILE}")
print(summary_df.to_string(index=False))


# ============================================================
# Create output Dataset
# ============================================================

out_ds = xr.Dataset(
    data_vars={
        "PP": (
            ("time", "region"),
            pp_mean,
            {
                "long_name": "Area-weighted regional mean primary productivity",
                "standard_name": "primary_productivity_of_biomass_expressed_as_carbon",
                "units": "mg m-2 day-1",
                "description": (
                    "Regional mean GlobColour PP calculated over the retained valid "
                    "RECCAP grid cells. The retained valid cells are defined by the "
                    "valid_region_mask, which excludes grid cells where the long-term "
                    "zero-filled percentage exceeds the chosen threshold. Within the "
                    "retained cells, monthly PP values that are NaN, negative, equal to "
                    "_FillValue, or equal to missing_value are replaced by zero before "
                    "area-weighted averaging. The denominator is the total retained "
                    "regional area weight, not the monthly valid-only area weight."
                ),
                "region_name": region_name_attr,
                "region_source": region_source_attr,
            },
        ),

        "PP_valid_only": (
            ("time", "region"),
            pp_valid_only_mean,
            {
                "long_name": "Area-weighted regional mean primary productivity using monthly valid-only denominator",
                "units": "mg m-2 day-1",
                "description": (
                    "Sensitivity estimate. Monthly invalid PP values are excluded from "
                    "both numerator and denominator. This differs from PP, where invalid "
                    "monthly values are set to zero and the denominator is the total "
                    "retained regional area weight."
                ),
            },
        ),

        "negative_grid_count": (
            ("time", "region"),
            negative_grid_count,
            {
                "long_name": "Number of retained PP grid cells with true negative values",
                "units": "count",
            },
        ),

        "nan_grid_count": (
            ("time", "region"),
            nan_grid_count,
            {
                "long_name": "Number of retained PP grid cells with NaN values",
                "units": "count",
            },
        ),

        "fill_value_grid_count": (
            ("time", "region"),
            fill_value_grid_count,
            {
                "long_name": "Number of retained PP grid cells with fill or missing values",
                "units": "count",
            },
        ),

        "zero_filled_grid_count": (
            ("time", "region"),
            zero_filled_grid_count,
            {
                "long_name": "Number of retained PP grid cells replaced by zero",
                "units": "count",
                "description": "NaN, true negative, fill-value, or missing-value PP cells.",
            },
        ),

        "negative_area_fraction": (
            ("time", "region"),
            negative_area_fraction,
            {
                "long_name": "Area fraction of retained PP grid cells with true negative values",
                "units": "1",
            },
        ),

        "nan_area_fraction": (
            ("time", "region"),
            nan_area_fraction,
            {
                "long_name": "Area fraction of retained PP grid cells with NaN values",
                "units": "1",
            },
        ),

        "fill_value_area_fraction": (
            ("time", "region"),
            fill_value_area_fraction,
            {
                "long_name": "Area fraction of retained PP grid cells with fill or missing values",
                "units": "1",
            },
        ),

        "zero_filled_area_fraction": (
            ("time", "region"),
            zero_filled_area_fraction,
            {
                "long_name": "Area fraction of retained PP grid cells replaced by zero",
                "units": "1",
            },
        ),

        "monthly_valid_area_fraction": (
            ("time", "region"),
            monthly_valid_area_fraction,
            {
                "long_name": "Monthly valid area fraction within retained region mask",
                "units": "1",
                "description": (
                    "Area fraction of retained grid cells that have valid PP in each month. "
                    "This is 1 - zero_filled_area_fraction."
                ),
            },
        ),

        "region_name": (
            ("region",),
            np.array(region_name_list, dtype=object),
            {
                "long_name": "RECCAP2 region name",
            },
        ),

        "basin": (
            ("region",),
            np.array(basin_list, dtype=object),
            {
                "long_name": "Original RECCAP2 basin mask variable",
            },
        ),

        "original_region_id": (
            ("region",),
            np.array(original_region_id_list, dtype=np.int16),
            {
                "long_name": "Original region ID in source RECCAP2 basin mask",
            },
        ),

        "n_total_cells_original_mask": (
            ("region",),
            n_total_cells_original_mask.astype(np.int32),
            {
                "long_name": "Number of grid cells in original all-region mask before zero-percent filtering",
            },
        ),

        "n_valid_cells_threshold_mask": (
            ("region",),
            n_valid_cells_threshold_mask.astype(np.int32),
            {
                "long_name": "Number of retained grid cells after zero-percent filtering",
            },
        ),

        "valid_fraction_percent_threshold_mask": (
            ("region",),
            valid_fraction_percent_threshold_mask.astype(np.float32),
            {
                "long_name": "Fraction of original region grid cells retained after zero-percent filtering",
                "units": "%",
            },
        ),

        "area_weight_used_for_regional_mean": (
            ("region",),
            np.array([region_total_weight[int(c)] for c in region_codes], dtype=np.float64),
            {
                "long_name": "Latitude-weighted area denominator used for regional mean",
                "description": "Weights are proportional to cos(latitude).",
            },
        ),

        "n_cells_used_for_regional_mean": (
            ("region",),
            np.array([region_total_grid_count[int(c)] for c in region_codes], dtype=np.int32),
            {
                "long_name": "Number of retained grid cells used for regional mean",
            },
        ),
    },

    coords={
        "time": times,
        "region": region_codes,
    },

    attrs={
        "title": "GlobColour PP regional means over valid RECCAP2 all-region mask",
        "source_data_directory": str(DATA_DIR),
        "source_file_pattern": "PP_YYYY_MM.nc",
        "source_variable": "PP",
        "source_units": "mg m-2 day-1",
        "valid_mask_file": str(VALID_MASK_FILE),
        "valid_mask_variable": VALID_MASK_VAR,
        "period": f"{YEAR_START}-{YEAR_END}",
        "method": (
            "Regional means are calculated over RECCAP2 all-region grid cells retained "
            "by valid_region_mask. The valid mask excludes grid cells where the long-term "
            "zero-filled percentage exceeds the chosen threshold. Within retained cells, "
            "monthly PP values that are NaN, true negative, equal to _FillValue, or equal "
            "to missing_value are replaced by zero before averaging. The denominator for "
            "PP is the total retained regional latitude-weighted area. PP_valid_only is "
            "also saved as a sensitivity estimate using only monthly valid cells in the "
            "denominator."
        ),
        "time_coordinate_note": (
            "The output time coordinate is based on the filename month and set to the 15th day "
            "of each month. The data values are from PP(time=0) in each source file."
        ),
        "created_by": "custom Python/xarray script",
        "date_created": date.today().isoformat(),
    },
)

out_ds["time"].attrs.update(
    {
        "long_name": "time",
        "description": "Monthly representative time based on filename, set to day 15.",
    }
)

out_ds["region"].attrs.update(
    {
        "long_name": "Valid RECCAP2 all-region code",
        "description": (
            "Only regions with at least one retained grid cell in valid_region_mask are included."
        ),
    }
)


# ============================================================
# Save NetCDF
# ============================================================

encoding = {
    "PP": {
        "dtype": "float32",
        "_FillValue": np.float32(-999.0),
        "zlib": True,
        "complevel": 4,
    },
    "PP_valid_only": {
        "dtype": "float32",
        "_FillValue": np.float32(-999.0),
        "zlib": True,
        "complevel": 4,
    },
    "negative_grid_count": {
        "dtype": "int32",
        "_FillValue": None,
        "zlib": True,
        "complevel": 4,
    },
    "nan_grid_count": {
        "dtype": "int32",
        "_FillValue": None,
        "zlib": True,
        "complevel": 4,
    },
    "fill_value_grid_count": {
        "dtype": "int32",
        "_FillValue": None,
        "zlib": True,
        "complevel": 4,
    },
    "zero_filled_grid_count": {
        "dtype": "int32",
        "_FillValue": None,
        "zlib": True,
        "complevel": 4,
    },
    "negative_area_fraction": {
        "dtype": "float32",
        "_FillValue": np.float32(-999.0),
        "zlib": True,
        "complevel": 4,
    },
    "nan_area_fraction": {
        "dtype": "float32",
        "_FillValue": np.float32(-999.0),
        "zlib": True,
        "complevel": 4,
    },
    "fill_value_area_fraction": {
        "dtype": "float32",
        "_FillValue": np.float32(-999.0),
        "zlib": True,
        "complevel": 4,
    },
    "zero_filled_area_fraction": {
        "dtype": "float32",
        "_FillValue": np.float32(-999.0),
        "zlib": True,
        "complevel": 4,
    },
    "monthly_valid_area_fraction": {
        "dtype": "float32",
        "_FillValue": np.float32(-999.0),
        "zlib": True,
        "complevel": 4,
    },
    "original_region_id": {
        "dtype": "int16",
        "_FillValue": None,
    },
    "n_total_cells_original_mask": {
        "dtype": "int32",
        "_FillValue": None,
    },
    "n_valid_cells_threshold_mask": {
        "dtype": "int32",
        "_FillValue": None,
    },
    "valid_fraction_percent_threshold_mask": {
        "dtype": "float32",
        "_FillValue": np.float32(-999.0),
    },
    "area_weight_used_for_regional_mean": {
        "dtype": "float64",
        "_FillValue": None,
    },
    "n_cells_used_for_regional_mean": {
        "dtype": "int32",
        "_FillValue": None,
    },
    "region": {
        "dtype": "int16",
        "_FillValue": None,
    },
    "time": {
        "dtype": "float64",
        "units": "days since 1900-01-01",
        "calendar": "gregorian",
    },
}

out_ds.to_netcdf(OUT_FILE, encoding=encoding, engine="netcdf4")

print(f"\nSaved NetCDF: {OUT_FILE}")
print(out_ds)


# ============================================================
# Save zero-fill log CSV
# ============================================================

log_df = pd.DataFrame(log_rows)

if len(log_df) > 0:
    log_df.to_csv(LOG_FILE, index=False)

    print(f"\nSaved zero-fill log: {LOG_FILE}")

    print("\nZero-filled cases summary:")
    print(
        log_df.groupby(["region", "basin", "original_region_id", "region_name"])[
            [
                "zero_filled_grid_count",
                "negative_grid_count",
                "nan_grid_count",
                "fill_value_grid_count",
            ]
        ]
        .sum()
        .reset_index()
        .to_string(index=False)
    )

else:
    print("\nNo negative, fill, or NaN PP values were found inside retained valid regions.")

Number of PP files found: 336
first files: /nird/datalake/NS11071K/users/yongyub/Observation/GlobColour/PP_1998_01.nc
first files: /nird/datalake/NS11071K/users/yongyub/Observation/GlobColour/PP_1998_02.nc
first files: /nird/datalake/NS11071K/users/yongyub/Observation/GlobColour/PP_1998_03.nc
last files : /nird/datalake/NS11071K/users/yongyub/Observation/GlobColour/PP_2025_10.nc
last files : /nird/datalake/NS11071K/users/yongyub/Observation/GlobColour/PP_2025_11.nc
last files : /nird/datalake/NS11071K/users/yongyub/Observation/GlobColour/PP_2025_12.nc

PP grid from valid mask:
lat: (4320,) -89.97917175292969 89.97916412353516
lon: (8640,) -179.9791717529297 179.9791717529297

Retained region codes:
[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 18 19 27 28 29]
Number of retained regions: 21

Retained region metadata:
 region       region_name    basin  original_region_id  n_total_cells_original_mask  n_valid_cells_threshold_mask  valid_fraction_percent_threshold_mask  region_fully_ex

/tmp/ipykernel_1866085/3559134258.py:629: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  out_ds = xr.Dataset(



Saved NetCDF: /nird/datalake/NS11071K/users/yongyub/Observation/GlobColour/RECCAP2_all_regions_valid_zero30_region_mean_PP/GlobColour_PP_RECCAP2_all_regions_valid_zero30_area_mean_1998_2025.nc
<xarray.Dataset>
Dimensions:                                (time: 336, region: 21)
Coordinates:
  * time                                   (time) datetime64[ns] 1998-01-15 ....
  * region                                 (region) int16 1 2 3 4 ... 27 28 29
Data variables: (12/19)
    PP                                     (time, region) float32 78.9 ... 299.8
    PP_valid_only                          (time, region) float32 186.9 ... 2...
    negative_grid_count                    (time, region) int32 0 0 0 ... 0 0 0
    nan_grid_count                         (time, region) int32 0 0 0 ... 0 0 0
    fill_value_grid_count                  (time, region) int32 325020 0 ... 0 0
    zero_filled_grid_count                 (time, region) int32 325020 0 ... 0 0
    ...                                  